Imports

In [ ]:
from pathlib import Path
import json
import csv
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from PIL import Image


Configuration

In [ ]:
PROCESSED_ROOT = Path("../../data/processed/Stage0")
INDEX_CSV      = PROCESSED_ROOT / "index.csv"

TRAIN_LABELS_JSON = Path("../../data/labels/Stage0/train.json")
VAL_LABELS_JSON   = Path("../../data/labels/Stage0/val.json")

NUM_CLASSES = 2
BATCH_SIZE  = 80
LR          = 1e-3
EPOCHS      = 15
IMAGE_SIZE  = 256

# Choose "rgb" (3-channel) or "gray" (1-channel)
IMAGE_MODE = "rgb"   # change to "gray" if you want grayscale training

MODEL_OUT_PATH = Path("../../models/classifier/Stage0/model.pt")
MODEL_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Load index.csv

In [ ]:
def read_index_csv(index_csv_path: Path):
    rows = []
    with open(index_csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append(r)
    if not rows:
        raise ValueError(f"index.csv is empty: {index_csv_path}")
    if "filepath" not in rows[0]:
        raise ValueError("index.csv must contain a 'filepath' column")
    return rows

index_rows = read_index_csv(INDEX_CSV)
print("Rows in index.csv:", len(index_rows))
print("Example row keys:", list(index_rows[0].keys()))
print("Example filepath:", index_rows[0]["filepath"])


Load label JSONs (train/val)

This expects a simple dict mapping:

{
  "images/train/xxx.png": 0,
  "images/train/yyy.png": 1
}

In [ ]:
def load_label_map(label_path: Path):
    """
    Stage0 tray detection:
    Assume ALL images in this dataset contain a tray (positive = 1).
    'No tray' negatives will be synthesized inside the Dataset.
    """
    return {}

train_label_map = load_label_map(TRAIN_LABELS_JSON)
val_label_map   = load_label_map(VAL_LABELS_JSON)

In [ ]:

from pathlib import Path
import cv2

# 🔧 set your image path
IMG_PATH = Path("../../data/processed/Stage0/images/train/2026-01-21_10-36-00-094_orig.png")

img = cv2.imread(str(IMG_PATH))
if img is None:
    raise FileNotFoundError(f"Could not read image: {IMG_PATH}")

# OpenCV uses BGR; display is fine anyway
print("Drag a rectangle over INNER tray area, then press ENTER (or SPACE). Press ESC to cancel.")
x, y, w, h = cv2.selectROI("Select TRAY_ROI", img, showCrosshair=True, fromCenter=False)
cv2.destroyAllWindows()

if w == 0 or h == 0: 
    raise RuntimeError("ROI selection cancelled or invalid.")

x0, y0, x1, y1 = int(x), int(y), int(x + w), int(y + h)
TRAY_ROI = (x0, y0, x1, y1)

print("\nTRAY_ROI =", TRAY_ROI)

# Preview crop + save
crop = img[y0:y1, x0:x1]
out_path = IMG_PATH.with_name(IMG_PATH.stem + "_roi_crop.png")
cv2.imwrite(str(out_path), crop)
print("Saved cropped preview to:", out_path)


In [ ]:
import random
import numpy as np
# 🔧 Tune these numbers once (tray interior)
TRAY_ROI = (57, 98, 1395, 969) 


def crop_tray_roi(img: Image.Image) -> Image.Image:
    return img.crop(TRAY_ROI)


def make_synthetic_negative(img: Image.Image) -> Image.Image:
    w, h = img.size
    cw = int(w * random.uniform(0.20, 0.35))
    ch = int(h * random.uniform(0.20, 0.35))
    x0 = random.randint(0, w - cw)
    y0 = random.randint(0, h - ch)
    patch = img.crop((x0, y0, x0 + cw, y0 + ch))
    patch = patch.resize((w, h), Image.BILINEAR)
    return patch

Define Dataset (respects split, no leakage)

In [ ]:
class ProcessedSplitDataset(Dataset):
    def __init__(self, index_rows, processed_root: Path, split: str, transform=None, p_neg=0.5):
        self.processed_root = processed_root
        self.split = split
        self.transform = transform
        self.p_neg = p_neg

        filepaths = []
        for r in index_rows:
            fp = r["filepath"].replace("\\", "/")
            row_split = (r.get("split") or "").strip().lower()
            if row_split:
                if row_split == split:
                    filepaths.append(fp)
            else:
                if f"images/{split}/" in fp:
                    filepaths.append(fp)

        if not filepaths:
            raise ValueError(f"No samples found for split='{split}'")

        self.filepaths = filepaths

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        rel_path = self.filepaths[idx]
        img_path = self.processed_root / rel_path

        img = Image.open(img_path)

        # mode
        img = img.convert("RGB") if IMAGE_MODE == "rgb" else img.convert("L")

        # focus on tray region
        img = crop_tray_roi(img)

        # label: 1 = tray present, 0 = synthetic "no tray"
        if random.random() < self.p_neg:
            img = make_synthetic_negative(img)
            label = 0
        else:
            label = 1

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

Transforms + DataLoaders

In [ ]:
import matplotlib.pyplot as plt

if IMAGE_MODE == "gray":
    transform = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),  # -> [1, H, W]
    ])
    in_channels = 1
else:
    transform = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),  # -> [3, H, W]
    ])
    in_channels = 3

train_ds = ProcessedSplitDataset(index_rows, PROCESSED_ROOT, "train", transform=transform, p_neg=0.5)
val_ds   = ProcessedSplitDataset(index_rows, PROCESSED_ROOT, "val",   transform=transform, p_neg=0.5)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def show_dataset_samples(ds, n=6):
    plt.figure(figsize=(15, 4))
    for i in range(n):
        img, label = ds[i]
        img = img.permute(1, 2, 0).numpy() # CHW → HWC


        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.title("TRAY" if label.item() == 1 else "NO TRAY")
        plt.axis("off")
    plt.show()


# visualize
show_dataset_samples(train_ds, n=8)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

# Quick check tensor shapes
x, y = next(iter(train_loader))
print("Batch image shape:", x.shape, "| Batch label shape:", y.shape)


Define a simple CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # /2
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),          # /4
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),          # /8
        )
        # IMAGE_SIZE is global; compute flattened size dynamically
        # create a "fake" image -> pass to features -> measure output size automatically
        # TO AVOID HARDCODING SIZES AND BREAKING MODEL WHEN IMAGE_SIZE CHANGES
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, IMAGE_SIZE, IMAGE_SIZE)
            out = self.features(dummy)
            flat_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(flat_dim, 128), nn.ReLU(),
            nn.Dropout(0.2), # turn off 20% of neurons, prevent memorisation, prevent double-descent peak
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)
    

# LESSER PARAMETER (BETTER FOR SMALL DATASET)
class SimpleCNN_GAP(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )


        # Global Average Pool: (B, 64, H, W) -> (B, 64, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))


        self.classifier = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )


    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1) # (B, 64)
        return self.classifier(x)
    

#model = SimpleCNN(in_channels=in_channels, num_classes=NUM_CLASSES).to(device)
model = SimpleCNN_GAP(in_channels=in_channels,num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss() # Used for classification, compare predicted scores vs true label
optimizer = optim.Adam(model.parameters(), lr=LR) # Smart gradient descent, adjust learning rates automatically

print(model)


In [ ]:
import torch.nn as nn

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params

count_parameters(model)

Training Loop (train + val)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / max(1, train_total)
    train_acc  = train_correct / max(1, train_total)

    # ---- Val ----
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            logits = model(imgs)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / max(1, val_total)
    val_acc  = val_correct / max(1, val_total)

    print(
        f"Epoch [{epoch}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f} "
        f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}"
    )


Save Model

In [ ]:
torch.save(model.state_dict(), MODEL_OUT_PATH)
print("Saved model to:", MODEL_OUT_PATH)
